# Notebook 03 — ALS Training + Evaluation + Serving Artifacts (WBS 3.4, 3.5)

Input: curated parquet + split cutoffs (chạy Notebook 01 TRƯỚC để có `split_stats.csv`).
Output: `als_topn.json`, `metrics.csv` (RMSE + Recall@K/NDCG@K), MODEL_DESIGN inputs.

**Gates:** KILL-LEAKAGE (split lại như notebook 01) · KILL-METRIC (band [0.6, 1.1],
so sánh ALS vs MovieMean) · KILL-CONTRACT (als_topn schema §3.3, no already-rated) ·
Reproducible (checkpoint dir + fixed config).
ALS grid (justified — Spark docs + practice, chi tiết PLAN §research):
rank ∈ {10, 50}, regParam ∈ {0.05, 0.1, 0.17}, maxIter = 15, nonnegative = True.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/movielens32m'
CURATED = f'{BASE}/curated'; ARTIFACTS = f'{BASE}/artifacts'; EVID = f'{BASE}/evidence'; CKPT = f'{BASE}/checkpoints'
import os
for d in [ARTIFACTS, EVID, CKPT]: os.makedirs(d, exist_ok=True)

import os
os.environ['SPARK_DRIVER_MEMORY'] = '8g'
!dpkg -l | grep -q openjdk-11 || (apt-get update -qq && apt-get install -qq -y openjdk-11-jre-headless)
import glob as _g
_jvm = sorted(_g.glob('/usr/lib/jvm/java-11*'))
assert _jvm, 'ERR: openjdk-11 not installed — run !apt-get install -y openjdk-11-jre-headless and retry'
os.environ['JAVA_HOME'] = _jvm[0]
print('JAVA_HOME ->', os.environ['JAVA_HOME'])
!pip install -q pyspark==3.5.7
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master('local[*]').appName('als')
         .config('spark.driver.memory', '6g')   # 6g heap: để ~6GB cho Python/OS trên Colab 12.7GB (cache tự spill đĩa nếu thiếu).config('spark.sql.shuffle.partitions', '32').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
spark.sparkContext.setCheckpointDir(CKPT)   # MANDATORY — ALS OOM guard
ratings = spark.read.parquet(f'{CURATED}/curated_ratings').cache()
print(f'ratings: {ratings.count():,}')

In [ ]:
# Recreate exact temporal split from Notebook 01 cutoffs (single source of truth)
import pandas as pd
split_stats = pd.read_csv(f'{EVID}/split_stats.csv')
cut_val, cut_test = split_stats['cut_val'][0], split_stats['cut_test'][0]
train = ratings.filter(f'rating_ts < {cut_val}').cache()
val = ratings.filter(f'rating_ts >= {cut_val} AND rating_ts < {cut_test}').cache()
test = ratings.filter(f'rating_ts >= {cut_test}').cache()
max_tr = train.agg({'rating_ts': 'max'}).first()[0]
assert max_tr < val.agg({'rating_ts': 'min'}).first()[0] < test.agg({'rating_ts': 'min'}).first()[0], 'KILL-LEAKAGE'
print(f'split OK: train={train.count():,} val={val.count():,} test={test.count():,}')

In [ ]:
# ALS grid search on VALIDATION RMSE (KILL-METRIC band [0.6, 1.1])
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
als_eval = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')
GRID = [(rank, reg) for rank in [10, 50] for reg in [0.05, 0.1, 0.17]]
results = []
best = None
for rank, reg in GRID:
    als = (ALS(userCol='userId', itemCol='movieId', ratingCol='rating',
               rank=rank, maxIter=15, regParam=reg, nonnegative=True,
               coldStartStrategy='drop', seed=42))
    model = als.fit(train)
    rmse_val = als_eval.evaluate(model.transform(val))
    print(f'rank={rank:3d} reg={reg:5.2f} -> val RMSE = {rmse_val:.4f}')
    results.append({'rank': rank, 'regParam': reg, 'rmse_val': rmse_val})
    if best is None or rmse_val < best[0]: best = (rmse_val, rank, reg, als)
print(f'\nBEST: rank={best[1]}, regParam={best[2]}, val RMSE={best[0]:.4f}')
grid_df = pd.DataFrame(results)
grid_df.to_csv(f'{EVID}/als_grid_search.csv', index=False)
grid_df

In [ ]:
# Fit best config, evaluate on TEST, compare vs MovieMean baseline
best_als = ALS(userCol='userId', itemCol='movieId', ratingCol='rating',
               rank=best[1], maxIter=15, regParam=best[2], nonnegative=True,
               coldStartStrategy='drop', seed=42)
model = best_als.fit(train)
rmse_als_test = als_eval.evaluate(model.transform(test))

global_mean = train.agg(F.avg('rating')).first()[0]
movie_mean = train.groupBy('movieId').agg(F.avg('rating').alias('m_mean'))
mm_pred = (test.join(movie_mean, 'movieId', 'left')
           .withColumn('prediction', F.coalesce(F.col('m_mean'), F.lit(global_mean))))
rmse_mm_test = als_eval.evaluate(mm_pred)

BAND = (0.6, 1.1)
print(f'ALS test RMSE      = {rmse_als_test:.4f}')
print(f'MovieMean test RMSE = {rmse_mm_test:.4f}')
print(f'improvement: {100*(rmse_mm_test - rmse_als_test)/rmse_mm_test:.2f}%')
assert BAND[0] <= rmse_als_test <= BAND[1], f'KILL-METRIC: ALS RMSE {rmse_als_test:.4f} ngoài band {BAND}'
assert rmse_als_test < rmse_mm_test, 'KILL-METRIC: ALS không thắng baseline — kiểm tra config/leakage'

## Ranking evaluation — Recall@K, NDCG@K (K ∈ {10, 20}, relevant := rating ≥ 4.0)
Ground truth: movies rating ≥ 4.0 trong **test holdout** của mỗi user.
Đối chiếu 2 nguồn chính: ALS Top-N vs Popularity Top-N (Content-Based per-user ranking = OPTIONAL, ghi rõ nếu bỏ).
(Band base-rate từ EDA: 49.81% ratings ≥ 4.0 → Recall cao không bất thường, phải disclose.)

In [ ]:
# Relevant test items per user
RELEVANT_THRESHOLD = 4.0; K_LIST = [10, 20]
relevant = (test.filter(F.col('rating') >= RELEVANT_THRESHOLD)
            .groupBy('userId').agg(F.collect_set('movieId').alias('rel_set')).cache())
n_rel_users = relevant.count()
print(f'users with >=1 relevant test item: {n_rel_users:,}')

# Candidate lists
movies_meta = spark.read.parquet(f'{CURATED}/curated_movies').select('movieId', 'title', 'genres')
pop_train = (train.groupBy('movieId').agg(F.count('*').alias('support'), F.avg('rating').alias('avg_r')))
POP_TOP = [r['movieId'] for r in pop_train.filter('support >= 100')
           .orderBy(F.desc('avg_r'), F.desc('support'), F.asc('movieId')).limit(20).collect()]
als_top = (model.recommendForAllUsers(20).select('userId',
           F.explode('recommendations').alias('rec')).select('userId', 'rec.movieId').alias('userId', 'movieId'))
# content-based per-user: top similar của phim user rated >= 4.0 trong TRAIN
liked_train = train.filter('rating >= 4.0').select('userId', 'movieId')
similar_map = spark.read.json(f'{ARTIFACTS}/similar_movies.json') if False else None
print('pop top20 ready; als candidates per user ready')

In [ ]:
# Compute Recall@K / NDCG@K for 2 sources (ALS, Popularity) — hit-based Recall, standard NDCG
import numpy as np
# Arrow-enabled toPandas (gọn RAM), chỉ user có relevant test item
rel_pd = relevant.select('userId', 'rel_set').toPandas()
rel_map = {int(u): set(m) for u, m in zip(rel_pd['userId'], rel_pd['rel_set'])}
print(f'users with relevant items in rel_map: {len(rel_map):,}')

def recall_ndcg(lists, name):
    out = []
    for K in K_LIST:
        h = n = 0.0; ndcg_sum = 0.0
        for u, items in lists.items():
            rel = rel_map.get(u, [])
            if not rel: continue
            n += 1
            top = items[:K]
            inter = set(top) & set(rel)
            h += (1 if inter else 0)
            dcg = sum(1/np.log2(i + 2) for i, m in enumerate(top) if m in rel)
            idcg = sum(1/np.log2(i + 2) for i in range(min(len(rel), K)))
            ndcg_sum += (dcg/idcg if idcg else 0)
        out.append({'source': name, 'K': K, 'recall': h/n, 'ndcg': ndcg_sum/n, 'n_users': int(n)})
        print(f'{name} @{K}: Recall={h/n:.4f} NDCG={ndcg_sum/n:.4f} (n={int(n):,})')
    return out

# ALS candidates per user (from recommendForAllUsers)
spark.conf.set('spark.sql.execution.arrow.pyspark.enabled', 'true')
print('collecting ALS candidates (~12M rows via Arrow)...')
als_pd = als_top.toPandas()
als_lists = als_pd.groupby('userId')['movieId'].apply(list).to_dict()
pop_lists = {u: POP_TOP for u in rel_map}   # popularity giống mọi user
rank_df = pd.DataFrame(recall_ndcg(als_lists, 'ALS') + recall_ndcg(pop_lists, 'Popularity'))
rank_df

In [ ]:
# GATE KILL-METRIC (ranking): Recall@K = 1.0 trên toàn bộ user ⟹ methodology lỗi
for _, r in rank_df.iterrows():
    assert r['recall'] < 1.0, f'KILL-METRIC: Recall@{int(r["K"])} = 1.0 — điều tra (leakage?)'
print('ranking sanity: all Recall@K < 1.0 OK')

# Save metrics.csv (full evaluation table — B3.5 deliverable)
metrics = pd.DataFrame([
    {'task': 'rating_prediction', 'model': 'MovieMean', 'metric': 'RMSE', 'set': 'test', 'value': rmse_mm_test},
    {'task': 'rating_prediction', 'model': 'ALS', 'metric': 'RMSE', 'set': 'test', 'value': rmse_als_test},
])
for _, r in rank_df.iterrows():
    metrics = pd.concat([metrics, pd.DataFrame([{
        'task': 'topn_ranking', 'model': r['source'],
        'metric': f'Recall@{int(r["K"])}', 'set': 'test_holdout', 'value': r['recall']}])], ignore_index=True)
    metrics = pd.concat([metrics, pd.DataFrame([{
        'task': 'topn_ranking', 'model': r['source'],
        'metric': f'NDCG@{int(r["K"])}', 'set': 'test_holdout', 'value': r['ndcg']}])], ignore_index=True)
metrics.to_csv(f'{EVID}/metrics.csv', index=False)
print(metrics.to_string())

In [ ]:
# Precompute ALS Top-N artifact (contract §3.3): generate → remove already-rated → rank → top-N
N_TOP = 10
from pyspark.sql.window import Window
recs = model.recommendForAllUsers(N_TOP + 50).select('userId', F.explode('recommendations').alias('rec'))
recs = recs.select('userId', F.col('rec.movieId').alias('movieId'), F.col('rec.rating').alias('score'))
# remove movies user đã rate (trong TOÀN BỘ curated data — online layer check lại history mới)
rated = ratings.select('userId', 'movieId')
recs = recs.join(rated, ['userId', 'movieId'], 'left_anti')
w = Window.partitionBy('userId').orderBy(F.desc('score'), F.asc('movieId'))
recs = (recs.withColumn('rank', F.row_number().over(w)).filter(F.col('rank') <= N_TOP)
        .orderBy('userId', 'rank'))
n_users_served = recs.select('userId').distinct().count()
print(f'users with final ALS Top-{N_TOP}: {n_users_served:,} (coldStart drop: users chỉ có test ratings bị drop)')

import json, datetime
print('collecting final top-N (~2M rows)...')
recs_pd = recs.toPandas().groupby('userId')
docs = []
for uid, grp in recs_pd:
    docs.append({'userId': int(uid), 'modelVersion': 'v1.0.0',
                 'generatedAt': datetime.datetime.utcnow().isoformat() + 'Z',
                 'strategy': 'ALS',
                 'recommendations': [{'movieId': int(r['movieId']), 'score': round(float(r['score']), 3),
                                     'rank': int(r['rank'])} for _, r in grp.iterrows()]})
with open(f'{ARTIFACTS}/als_topn.json', 'w') as f: json.dump(docs, f)
print('als_topn.json saved — M2 handoff artifact')

In [ ]:
# GATE KILL-CONTRACT: full validation (NOT sample — full check via Spark join, no driver OOM)
sample = docs[0]
assert set(sample.keys()) == {'userId', 'modelVersion', 'generatedAt', 'strategy', 'recommendations'}
assert sample['strategy'] == 'ALS' and len(sample['recommendations']) <= 10
assert all(r['rank'] == i + 1 for i, r in enumerate(sample['recommendations']))

# no already-rated: re-check FULL artifact via Spark join (left_anti must keep all rows)
rated_df = ratings.select('userId', 'movieId')
n_recs = recs.count()
n_after_anti = recs.join(rated_df, ['userId', 'movieId'], 'left_anti').count()
assert n_recs == n_after_anti, f'KILL-CONTRACT: {n_recs - n_after_anti} already-rated items leaked into als_topn'
print(f'KILL-CONTRACT PASS: {n_recs:,} recommendations, 0 already-rated (full check, not sample)')
print('\n=== M2 ARTIFACTS READY: popular_movies.json, similar_movies.json, als_topn.json, metrics.csv ===')

In [ ]:
# PERSIST for Person 2 (L2B serving + retrain/promotion B6):
#  (1) ALS model saved to Drive   (2) user_history SEED parquet (Person 2 aggregates into §3.4 docs)
#  (3) model_card.json = single source of truth for downstream
MODELS = f'{BASE}/models'
os.makedirs(MODELS, exist_ok=True)
MODEL_DIR = f'{MODELS}/als_v1.0.0'
model.write().overwrite().save(MODEL_DIR)   # Spark ML format: Person 2 loads via ALSModel.load(MODEL_DIR)
print(f'ALS model saved: {MODEL_DIR}')

# user_history SEED: raw ratings Person 2 aggregates INTO contract §3.4 user_history docs
# (interaction_count, recent_movieIds, positive_movieIds, lastUpdated) at Mongo import time
(ratings.select('userId', 'movieId', 'rating', 'rating_ts')
 .write.mode('overwrite').parquet(f'{ARTIFACTS}/user_history_seed.parquet'))
n_hist = ratings.count()
print(f'user_history_seed.parquet: {n_hist:,} rows')

# model_card.json — everything Person 2 needs without re-reading this notebook
model_card = {
    'modelVersion': 'v1.0.0',
    'modelType': 'ALS (implicit feedback off, explicit ratings)',
    'config': {'rank': int(best[1]), 'regParam': float(best[2]), 'maxIter': 15,
                'nonnegative': True, 'coldStartStrategy': 'drop', 'seed': 42},
    'metrics': {'rmse_val': float(best[0]), 'rmse_test': float(rmse_als_test),
                 'rmse_movemean_test': float(rmse_mm_test)},
    'split': {'method': 'global_temporal_70_15_15',
              'cut_val': str(cut_val), 'cut_test': str(cut_test)},
    'artifacts': {'als_topn': f'{ARTIFACTS}/als_topn.json',
                  'popular_movies': f'{ARTIFACTS}/popular_movies.json',
                  'similar_movies': f'{ARTIFACTS}/similar_movies.json',
                  'user_history_seed': f'{ARTIFACTS}/user_history_seed.parquet',
                  'als_model_dir': MODEL_DIR},
    'generatedAt': datetime.datetime.utcnow().isoformat() + 'Z',
}
with open(f'{ARTIFACTS}/model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)
print(json.dumps(model_card['config'], indent=2))
print('\n=== FULL M2 PACKAGE: 4 artifacts + ALS model + model_card.json ===')

## ✅ Notebook 03 DONE khi (MILESTONE M2 — handoff Person 2):
- Band RMSE + ALS thắng baseline (đã assert)
- Recall@K < 1.0 sanity (đã assert)
- als_topn pass contract + no already-rated (đã assert)
- ALS model + user_history_seed.parquet + model_card.json đã persist trên Drive
- Copy về repo: `artifacts/*.json`, `evidence/{als_grid_search,metrics}.csv` (model + user_history ở lại Drive — quá lớn cho repo)
- Cập nhật CHECKLIST (mục 9–13 Done) + WORKLOG + MODEL_DESIGN.md từ grid results
- **GHI MODEL_DESIGN:** ALS config chọn + justification từ `als_grid_search.csv` (INV6)

**Person 2 nhận gì (M2 handoff):**
| File | Dùng để làm gì |
|------|----------------|
| `popular_movies.json` | Serving tier 0-history |
| `similar_movies.json` | Serving tier few-history (Mongo similar_movies collection) |
| `als_topn.json` | Serving tier enough-history (Mongo als_topn collection) |
| `user_history_seed.parquet` | Mongo user_history import + routing tier check |
| `models/als_v1.0.0/` (Drive) | B6.1 retrain baseline + promotion gate so sánh |
| `model_card.json` | Metadata chuẩn (version, config, metrics, split) — không phải hỏi lại Person 1 |